In [1]:
import sys
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoTokenizer

print("PyTorch version:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

PyTorch version: 2.14.0
Device: cpu


In [12]:
PROJECT_ROOT = Path.cwd().parent
REACTIONT5_DIR = PROJECT_ROOT / "external" / "ReactionT5v2"
TASK_FORWARD_DIR = REACTIONT5_DIR / "task_forward"

sys.path.insert(0, str(REACTIONT5_DIR))
sys.path.insert(0, str(TASK_FORWARD_DIR))

print("Project root:", PROJECT_ROOT)
print("ReactionT5 directory:", REACTIONT5_DIR)
print("Task forward directory:", TASK_FORWARD_DIR)
print("ReactionT5 found:", REACTIONT5_DIR.exists())
print("Task forward found:", TASK_FORWARD_DIR.exists())

Project root: /Users/tanyagoel101/Downloads/data-discovery
ReactionT5 directory: /Users/tanyagoel101/Downloads/data-discovery/external/ReactionT5v2
Task forward directory: /Users/tanyagoel101/Downloads/data-discovery/external/ReactionT5v2/task_forward
ReactionT5 found: True
Task forward found: True


In [5]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "sagawa/ReactionT5v2-forward"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("ReactionT5 loaded successfully!")
print("Device:", device)

/Users/tanyagoel101/miniconda3/envs/reactiont5/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


ReactionT5 loaded successfully!
Device: cpu


In [8]:
team_b_test = pd.DataFrame({
    "Target Left": [
        "8 Eu + 15 Ga + 30 Ge + 1 Mn",
        "86 Al + 2 Co + 1.5 La + 6 Ni + 4.5 Y",
        "0.5 Bi2O3 + 0.5 Fe2O3",
        "1 Al2O3 + 1 SrCO3",
        "52.5 Cu + 6 Ni + 30 Ti + 11.5 Zr"
    ],
    
    "Target Right": [
        "1 Eu8Ga15Mn1Ge30",
        "1 Al86Ni6Y4.5Co2La1.5",
        "1 BiFeO3",
        "1 SrAl2O4 + 1 CO2",
        "1 Cu52.5Ti30Zr11.5Ni6"
    ]
})

team_b_test

,Target Left,Target Right
0,8 Eu + 15 Ga + 30 Ge + 1 Mn,1 Eu8Ga15Mn1Ge30
1,86 Al + 2 Co + 1.5 La + 6 Ni + 4.5 Y,1 Al86Ni6Y4.5Co2La1.5
2,0.5 Bi2O3 + 0.5 Fe2O3,1 BiFeO3
3,1 Al2O3 + 1 SrCO3,1 SrAl2O4 + 1 CO2
4,52.5 Cu + 6 Ni + 30 Ti + 11.5 Zr,1 Cu52.5Ti30Zr11.5Ni6


In [13]:
from train import preprocess_df

reactiont5_input = pd.DataFrame({
    "REACTANT": team_b_test["Target Left"],
    "PRODUCT": team_b_test["Target Right"]
})

reactiont5_input

,REACTANT,PRODUCT
0,8 Eu + 15 Ga + 30 Ge + 1 Mn,1 Eu8Ga15Mn1Ge30
1,86 Al + 2 Co + 1.5 La + 6 Ni + 4.5 Y,1 Al86Ni6Y4.5Co2La1.5
2,0.5 Bi2O3 + 0.5 Fe2O3,1 BiFeO3
3,1 Al2O3 + 1 SrCO3,1 SrAl2O4 + 1 CO2
4,52.5 Cu + 6 Ni + 30 Ti + 11.5 Zr,1 Cu52.5Ti30Zr11.5Ni6


In [14]:
processed_test = preprocess_df(
    reactiont5_input.copy(),
    drop_duplicates=False
)

processed_test

,REACTANT,PRODUCT,CATALYST,REAGENT,SOLVENT,input
0,8 Eu + 15 Ga + 30 Ge + 1 Mn,1 Eu8Ga15Mn1Ge30,,,,REACTANT:8 Eu + 15 Ga + 30 Ge + 1 MnREAGENT:
1,86 Al + 2 Co + 1.5 La + 6 Ni + 4.5 Y,1 Al86Ni6Y4.5Co2La1.5,,,,REACTANT:86 Al + 2 Co + 1.5 La + 6 Ni + 4.5 YR...
2,0.5 Bi2O3 + 0.5 Fe2O3,1 BiFeO3,,,,REACTANT:0.5 Bi2O3 + 0.5 Fe2O3REAGENT:
3,1 Al2O3 + 1 SrCO3,1 SrAl2O4 + 1 CO2,,,,REACTANT:1 Al2O3 + 1 SrCO3REAGENT:
4,52.5 Cu + 6 Ni + 30 Ti + 11.5 Zr,1 Cu52.5Ti30Zr11.5Ni6,,,,REACTANT:52.5 Cu + 6 Ni + 30 Ti + 11.5 ZrREAGE...


In [15]:
def predict_reaction(input_text, num_predictions=5):
    # Tokenize the preprocessed ReactionT5 input
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=400,
        truncation=True
    ).to(device)

    # Generate ranked predictions
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=num_predictions,
            num_return_sequences=num_predictions,
            return_dict_in_generate=True,
            output_scores=True,
            max_length=400
        )

    # Convert token IDs back into predicted product strings
    predictions = tokenizer.batch_decode(
        outputs.sequences,
        skip_special_tokens=True
    )

    # Generation scores
    scores = outputs.sequences_scores.cpu().tolist()

    return list(zip(predictions, scores))

In [16]:
test_row = processed_test.iloc[2]

print("Model input:")
print(test_row["input"])

print("\nKnown product:")
print(test_row["PRODUCT"])

predictions = predict_reaction(test_row["input"])

print("\nReactionT5 predictions:")
for rank, (prediction, score) in enumerate(predictions, start=1):
    print(f"{rank}. {prediction}   score={score:.4f}")

Model input:
REACTANT:0.5 Bi2O3 + 0.5 Fe2O3REAGENT: 

Known product:
1 BiFeO3

ReactionT5 predictions:
1. [+3]. [+3]. [O-2]. [O-2]. [O-2]   score=-0.0459
2. []. []   score=-0.0709
3. [+3]. [+5]. [O-2]. [O-2]. [O-2]   score=-0.0851
4. [+5]. [+5]. [O-2]. [O-2]. [O-2]   score=-0.0953
5. 112OP 3(O1)OP (=O)(O2)O3   score=-0.0955


In [18]:
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")

prediction_rows = []

for rank, (prediction, score) in enumerate(predictions, start=1):
    mol = Chem.MolFromSmiles(prediction)
    
    prediction_rows.append({
        "Rank": rank,
        "Prediction": prediction,
        "Score": score,
        "Valid SMILES": mol is not None
    })

prediction_table = pd.DataFrame(prediction_rows)

prediction_table

,Rank,Prediction,Score,Valid SMILES
0,1,[+3]. [+3]. [O-2]. [O-2]. [O-2],-0.045920,False
1,2,[]. [],-0.070851,False
2,3,[+3]. [+5]. [O-2]. [O-2]. [O-2],-0.085098,False
3,4,[+5]. [+5]. [O-2]. [O-2]. [O-2],-0.095326,False
4,5,112OP 3(O1)OP (=O)(O2)O3,-0.095549,False


In [19]:
all_results = []

for i, row in processed_test.iterrows():
    predictions = predict_reaction(row["input"])

    for rank, (prediction, score) in enumerate(predictions, start=1):
        valid_smiles = Chem.MolFromSmiles(prediction) is not None

        all_results.append({
            "Reaction ID": i,
            "Target Left": row["REACTANT"],
            "Target Right": row["PRODUCT"],
            "Rank": rank,
            "Prediction": prediction,
            "Score": score,
            "Valid SMILES": valid_smiles
        })

results_df = pd.DataFrame(all_results)

results_df

,Reaction ID,Target Left,Target Right,Rank,Prediction,Score,Valid SMILES
0,0,8 Eu + 15 Ga + 30 Ge + 1 Mn,1 Eu8Ga15Mn1Ge30,1,[+3]. [+5]. [+5]. [O-2]. [O-2]. [O-2],-0.022837,False
1,0,8 Eu + 15 Ga + 30 Ge + 1 Mn,1 Eu8Ga15Mn1Ge30,2,[+5]. [+5]. [+5]. [+5]. [O-2]. [O-2]. [O-2],-0.025296,False
2,0,8 Eu + 15 Ga + 30 Ge + 1 Mn,1 Eu8Ga15Mn1Ge30,3,[+5]. [+5]. [+5]. [+5]. [O-2]. [O-2]. [O-2]. [...,-0.058633,False
3,0,8 Eu + 15 Ga + 30 Ge + 1 Mn,1 Eu8Ga15Mn1Ge30,4,[+3]. [+5]. [+5]. [+5]. [+5]. [O-2]. [O-2]. [O-2],-0.069230,False
4,0,8 Eu + 15 Ga + 30 Ge + 1 Mn,1 Eu8Ga15Mn1Ge30,5,[+5]. [+5]. [+5]. [+5]. [+5]. [O-2]. [O-2]. [O-2],-0.070281,False
5,1,86 Al + 2 Co + 1.5 La + 6 Ni + 4.5 Y,1 Al86Ni6Y4.5Co2La1.5,1,[+3]. [+3]. [O-2]. [O-2]. [O-2],-0.011430,False
6,1,86 Al + 2 Co + 1.5 La + 6 Ni + 4.5 Y,1 Al86Ni6Y4.5Co2La1.5,2,[+3]. [+3]. [+3]. [+5]. [O-]P 1[O-]. [O-2]. [O...,-0.059502,False
7,1,86 Al + 2 Co + 1.5 La + 6 Ni + 4.5 Y,1 Al86Ni6Y4.5Co2La1.5,3,[+3]. [+3]. [+3]. [+3]. [O-2]. [O-2]. [O-2]. [...,-0.070046,False
8,1,86 Al + 2 Co + 1.5 La + 6 Ni + 4.5 Y,1 Al86Ni6Y4.5Co2La1.5,4,[+3]. [+3]. [+3]. [+3]. [O-2]. [O-2]. [O-2],-0.071336,False
9,1,86 Al + 2 Co + 1.5 La + 6 Ni + 4.5 Y,1 Al86Ni6Y4.5Co2La1.5,5,[+3]. [+3]. [+3]. [+3]. [O-2]. [O-2]. [O-2]. [...,-0.076959,False


In [20]:
summary = pd.DataFrame({
    "Metric": [
        "Reactions tested",
        "Predictions generated",
        "Valid SMILES",
        "Invalid SMILES",
        "Valid SMILES rate"
    ],
    "Value": [
        results_df["Reaction ID"].nunique(),
        len(results_df),
        results_df["Valid SMILES"].sum(),
        (~results_df["Valid SMILES"]).sum(),
        f'{results_df["Valid SMILES"].mean():.1%}'
    ]
})

summary

,Metric,Value
0,Reactions tested,5
1,Predictions generated,25
2,Valid SMILES,2
3,Invalid SMILES,23
4,Valid SMILES rate,8.0%
